# EcoShield AI — Final Test Karşılaştırması

Bu notebook Görev 7 kapsamında daha önce validation ile seçilmiş iki yaklaşımı ortak test splitinde **yalnızca bir kez** değerlendirir:

1. Görev 5 cascade referansı: Small Random Forest → HeavyCatBoost
2. Görev 6'da önceden seçilen final aday: `cat_d8_balanced`

Kurallar:

- Modeller yeniden eğitilmez.
- Threshold değerleri test sonucuna göre değiştirilmez.
- Test üzerinde model veya hiperparametre seçimi yapılmaz.
- D8 Balanced, test açılmadan önce belirlenmiş final model adayıdır.
- Test yalnızca nihai genelleme performansını raporlamak için kullanılır.


## 1. Ayarlar, paket kontrolü ve tek-sefer koruması

In [ ]:
QUICK_MODE = False
RANDOM_STATE = 42
INFERENCE_SAMPLE_SIZE = 10_000
INFERENCE_REPEATS = 5
ALLOW_REPEAT_TEST_EVALUATION = False
PRESELECTED_FINAL_TRIAL = "cat_d8_balanced"

import importlib.util
import json
import os
import sys
import time
from pathlib import Path

required = {
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "psutil": "psutil",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing = [pip for module, pip in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError("Eksik paketler: " + ", ".join(missing))

print("Önceden seçilmiş final model:", PRESELECTED_FINAL_TRIAL)
print("Test eşikleri değiştirilmeyecek.")


## 2. Importlar, proje yolları ve gerekli dosyalar

In [ ]:
import gc
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "common_preprocessing.py").exists():
        NOTEBOOK_DIR = candidate
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    raise FileNotFoundError("notebooks/common_preprocessing.py bulunamadı.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from common_preprocessing import (
    build_project_paths,
    find_project_root,
    get_or_create_profile_cache,
)

PROJECT_ROOT = find_project_root(Path.cwd())
PATHS = build_project_paths(PROJECT_ROOT)
MODEL_DIR = PROJECT_ROOT / "models"
PREDICTIONS_DIR = PATHS.outputs / "predictions"
FIGURES_DIR = PATHS.outputs / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

CASCADE_SELECTION_PATH = PATHS.metrics / "selected_cascade_pipeline.csv"
SINGLE_SELECTION_PATH = PATHS.metrics / "selected_single_heavy_model.csv"
CASCADE_LIGHT_MODEL_PATH = MODEL_DIR / "light" / "small_random_forest_light.joblib"
CASCADE_HEAVY_MODEL_PATH = MODEL_DIR / "heavy" / "catboost_heavy.joblib"
SINGLE_MODEL_PATH = MODEL_DIR / "heavy" / "optimized_single_heavy_model.joblib"
FINAL_METADATA_PATH = PATHS.metadata / "final_test_evaluation_metadata.json"

required_paths = [
    CASCADE_SELECTION_PATH,
    SINGLE_SELECTION_PATH,
    CASCADE_LIGHT_MODEL_PATH,
    CASCADE_HEAVY_MODEL_PATH,
    SINGLE_MODEL_PATH,
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Görev 7 için eksik dosyalar:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

if FINAL_METADATA_PATH.exists() and not ALLOW_REPEAT_TEST_EVALUATION:
    raise RuntimeError(
        "Final test değerlendirmesi daha önce tamamlanmış. "
        "Testi model seçimi için tekrar çalıştırmayın. "
        "Bilinçli teknik tekrar gerekiyorsa ALLOW_REPEAT_TEST_EVALUATION=True yapın."
    )

print("Proje kökü:", PROJECT_ROOT)
print("Başlangıç RAM:", f"{psutil.Process(os.getpid()).memory_info().rss / (1024**3):.2f} GB")


## 3. Validation ile sabitlenmiş seçimleri yükleme

Bu hücre yalnızca önceden kaydedilmiş model adlarını ve threshold değerlerini okur. Test sonucuna göre hiçbir seçim yapılmaz.


In [ ]:
cascade_selection = pd.read_csv(CASCADE_SELECTION_PATH).iloc[0]
single_selection = pd.read_csv(SINGLE_SELECTION_PATH).iloc[0]

if cascade_selection["light_model"] != "SmallRandomForest":
    raise ValueError("Beklenen cascade hafif modeli SmallRandomForest değil.")
if cascade_selection["heavy_model"] != "HeavyCatBoost":
    raise ValueError("Beklenen cascade ağır modeli HeavyCatBoost değil.")
if single_selection["trial_name"] != PRESELECTED_FINAL_TRIAL:
    raise ValueError(
        f"Beklenen final deneme {PRESELECTED_FINAL_TRIAL}; "
        f"dosyada {single_selection['trial_name']} bulundu."
    )

cascade_light_threshold = float(cascade_selection["light_threshold"])
cascade_heavy_threshold = float(cascade_selection["heavy_threshold"])
single_threshold = float(single_selection["threshold"])

fixed_thresholds = pd.DataFrame([
    {
        "approach": "ValidationSelectedCascade",
        "model": "SmallRandomForest → HeavyCatBoost",
        "light_threshold": cascade_light_threshold,
        "heavy_or_single_threshold": cascade_heavy_threshold,
        "validation_precision": float(cascade_selection["precision"]),
        "validation_recall": float(cascade_selection["recall"]),
        "validation_f1": float(cascade_selection["f1"]),
        "validation_pr_auc": np.nan,
        "validation_roc_auc": np.nan,
    },
    {
        "approach": "PreselectedSingleHeavy",
        "model": PRESELECTED_FINAL_TRIAL,
        "light_threshold": np.nan,
        "heavy_or_single_threshold": single_threshold,
        "validation_precision": float(single_selection["precision"]),
        "validation_recall": float(single_selection["recall"]),
        "validation_f1": float(single_selection["f1"]),
        "validation_pr_auc": float(single_selection["validation_pr_auc"]),
        "validation_roc_auc": float(single_selection["validation_roc_auc"]),
    },
])

print("Sabit thresholdlar:")
display(fixed_thresholds)


## 4. Ortak test splitini model profillerinden yükleme

Test, IEEE-CIS resmi etiketsiz competition test dosyası değildir. Görev 1'de etiketli train verisinden ayrılmış ortak yerel test splitidir.


In [ ]:
load_progress = tqdm(total=2, desc="Test cache yükleme", unit=" profil")

tree_data = get_or_create_profile_cache(
    "random_forest", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=True,
)
tree_X_test = tree_data["X_test"]
y_test = np.asarray(tree_data["y_test"], dtype=np.int8)
id_test = np.asarray(tree_data["id_test"])
del tree_data
gc.collect()
load_progress.update(1)
load_progress.set_postfix(profile="sklearn_tree")

catboost_data = get_or_create_profile_cache(
    "catboost", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=True,
)
catboost_X_test = catboost_data["X_test"]
if not np.array_equal(id_test, catboost_data["id_test"]):
    raise ValueError("Tree ve CatBoost test TransactionID dizileri uyuşmuyor.")
if not np.array_equal(y_test, catboost_data["y_test"]):
    raise ValueError("Tree ve CatBoost test hedef dizileri uyuşmuyor.")
del catboost_data
gc.collect()
load_progress.update(1)
load_progress.set_postfix(profile="catboost")
load_progress.close()

if len(np.unique(id_test)) != len(id_test):
    raise ValueError("Test TransactionID dizisinde tekrar var.")
if set(np.unique(y_test)) != {0, 1}:
    raise ValueError("Test hedefi ikili değil.")

print("Test satırı:", len(y_test))
print("Test fraud sayısı:", int(y_test.sum()))
print("Tree test:", tree_X_test.shape)
print("CatBoost test:", catboost_X_test.shape)
print("RAM:", f"{psutil.Process(os.getpid()).memory_info().rss / (1024**3):.2f} GB")


## 5. Kaydedilmiş modelleri yükleme

In [ ]:
model_load_started = time.perf_counter()
cascade_light_model = joblib.load(CASCADE_LIGHT_MODEL_PATH)
cascade_heavy_model = joblib.load(CASCADE_HEAVY_MODEL_PATH)
single_model = joblib.load(SINGLE_MODEL_PATH)
model_load_seconds = time.perf_counter() - model_load_started

print(f"Üç model {model_load_seconds:.2f} sn içinde yüklendi.")
print("RAM:", f"{psutil.Process(os.getpid()).memory_info().rss / (1024**3):.2f} GB")


## 6. Sabit cascade ile test inference

Hafif model bütün test satırlarında çalışır. Yalnızca validation'da sabitlenmiş hafif eşiği geçen satırlar Task 3 HeavyCatBoost modeline gönderilir.


In [ ]:
cascade_started = time.perf_counter()
light_test_probabilities = cascade_light_model.predict_proba(tree_X_test)[:, 1]
routed_mask = light_test_probabilities >= cascade_light_threshold
routed_indices = np.flatnonzero(routed_mask)

heavy_routed_probabilities = cascade_heavy_model.predict_proba(
    catboost_X_test.iloc[routed_indices]
)[:, 1]
cascade_probabilities = np.zeros(len(y_test), dtype=np.float64)
cascade_probabilities[routed_indices] = heavy_routed_probabilities
cascade_predictions = (
    routed_mask & (cascade_probabilities >= cascade_heavy_threshold)
).astype(np.int8)
cascade_full_inference_seconds = time.perf_counter() - cascade_started

print("Ağır modele yönlendirilen:", f"{len(routed_indices):,} / {len(y_test):,}")
print("Yönlendirme oranı:", f"{routed_mask.mean():.2%}")
print("Cascade tam test inference:", f"{cascade_full_inference_seconds:.4f} sn")


## 7. Önceden seçilmiş D8 Balanced ile test inference

In [ ]:
single_started = time.perf_counter()
single_probabilities = single_model.predict_proba(catboost_X_test)[:, 1]
single_predictions = (single_probabilities >= single_threshold).astype(np.int8)
single_full_inference_seconds = time.perf_counter() - single_started

print("Tekil model tam test inference:", f"{single_full_inference_seconds:.4f} sn")


## 8. Gerçek 10K inference tekrar ölçümü

In [ ]:
sample_size = min(INFERENCE_SAMPLE_SIZE, len(y_test))
tree_sample = tree_X_test[:sample_size]
catboost_sample = catboost_X_test.iloc[:sample_size]


def measure_single_inference():
    _ = single_model.predict_proba(catboost_sample)[:, 1]
    durations = []
    for _ in tqdm(range(INFERENCE_REPEATS), desc="Tekil 10K inference", leave=False):
        started = time.perf_counter()
        _ = single_model.predict_proba(catboost_sample)[:, 1]
        durations.append(time.perf_counter() - started)
    return float(np.mean(durations)), float(np.std(durations))


def measure_cascade_inference():
    light_probs = cascade_light_model.predict_proba(tree_sample)[:, 1]
    sample_routed = light_probs >= cascade_light_threshold
    sample_indices = np.flatnonzero(sample_routed)
    if sample_indices.size:
        _ = cascade_heavy_model.predict_proba(
            catboost_sample.iloc[sample_indices]
        )[:, 1]

    durations = []
    routed_rates = []
    for _ in tqdm(range(INFERENCE_REPEATS), desc="Cascade 10K inference", leave=False):
        started = time.perf_counter()
        light_probs = cascade_light_model.predict_proba(tree_sample)[:, 1]
        sample_routed = light_probs >= cascade_light_threshold
        sample_indices = np.flatnonzero(sample_routed)
        if sample_indices.size:
            _ = cascade_heavy_model.predict_proba(
                catboost_sample.iloc[sample_indices]
            )[:, 1]
        durations.append(time.perf_counter() - started)
        routed_rates.append(sample_routed.mean())
    return (
        float(np.mean(durations)),
        float(np.std(durations)),
        float(np.mean(routed_rates)),
    )


cascade_10k_mean, cascade_10k_std, cascade_10k_routed_rate = (
    measure_cascade_inference()
)
single_10k_mean, single_10k_std = measure_single_inference()

print(
    f"Cascade 10K: {cascade_10k_mean:.4f} ± {cascade_10k_std:.4f} sn | "
    f"routed={cascade_10k_routed_rate:.2%}"
)
print(f"Tekil D8 10K: {single_10k_mean:.4f} ± {single_10k_std:.4f} sn")


## 9. Final test metrikleri

In [ ]:
def calculate_test_metrics(
    approach, model_name, probabilities, predictions,
    full_inference_seconds, inference_10k_mean, inference_10k_std,
    validation_row, routed_rate=1.0,
):
    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()
    test_precision = float(
        precision_score(y_test, predictions, zero_division=0)
    )
    test_recall = float(recall_score(y_test, predictions, zero_division=0))
    test_f1 = float(f1_score(y_test, predictions, zero_division=0))
    test_pr_auc = float(average_precision_score(y_test, probabilities))
    test_roc_auc = float(roc_auc_score(y_test, probabilities))
    return {
        "approach": approach,
        "model": model_name,
        "threshold_source": "validation",
        "model_selected_before_test": approach == "PreselectedSingleHeavy",
        "precision": test_precision,
        "recall": test_recall,
        "f1": test_f1,
        "pr_auc": test_pr_auc,
        "roc_auc": test_roc_auc,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_fraud_count": int(predictions.sum()),
        "routed_rate": float(routed_rate),
        "heavy_call_reduction": float(1.0 - routed_rate),
        "full_test_inference_seconds": float(full_inference_seconds),
        "inference_seconds_10k_mean": float(inference_10k_mean),
        "inference_seconds_10k_std": float(inference_10k_std),
        "validation_precision": float(validation_row["validation_precision"]),
        "validation_recall": float(validation_row["validation_recall"]),
        "validation_f1": float(validation_row["validation_f1"]),
        "precision_test_minus_validation": (
            test_precision - float(validation_row["validation_precision"])
        ),
        "recall_test_minus_validation": (
            test_recall - float(validation_row["validation_recall"])
        ),
        "f1_test_minus_validation": (
            test_f1 - float(validation_row["validation_f1"])
        ),
    }


cascade_metrics = calculate_test_metrics(
    "ValidationSelectedCascade",
    "SmallRandomForest → HeavyCatBoost",
    cascade_probabilities,
    cascade_predictions,
    cascade_full_inference_seconds,
    cascade_10k_mean,
    cascade_10k_std,
    fixed_thresholds.iloc[0],
    routed_rate=float(routed_mask.mean()),
)
single_metrics = calculate_test_metrics(
    "PreselectedSingleHeavy",
    PRESELECTED_FINAL_TRIAL,
    single_probabilities,
    single_predictions,
    single_full_inference_seconds,
    single_10k_mean,
    single_10k_std,
    fixed_thresholds.iloc[1],
    routed_rate=1.0,
)
final_test_comparison = pd.DataFrame([cascade_metrics, single_metrics])

display(final_test_comparison)
print(
    "Not: Bu tablo test üzerinde yeni model veya threshold seçmek için "
    "kullanılmayacaktır."
)


## 10. Final test görselleri

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for axis, metrics, title in [
    (axes[0], cascade_metrics, "Cascade"),
    (axes[1], single_metrics, "D8 Balanced"),
]:
    matrix = np.array([
        [metrics["tn"], metrics["fp"]],
        [metrics["fn"], metrics["tp"]],
    ])
    image = axis.imshow(matrix, cmap="Blues")
    for row in range(2):
        for column in range(2):
            axis.text(
                column, row, f"{matrix[row, column]:,}",
                ha="center", va="center", fontsize=13,
            )
    axis.set_xticks([0, 1], ["Normal", "Fraud"])
    axis.set_yticks([0, 1], ["Normal", "Fraud"])
    axis.set_xlabel("Tahmin")
    axis.set_ylabel("Gerçek")
    axis.set_title(title)
fig.suptitle("Final Test Confusion Matrisleri")
fig.tight_layout()
confusion_figure_path = FIGURES_DIR / "final_test_confusion_matrices.png"
fig.savefig(confusion_figure_path, dpi=170, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
PrecisionRecallDisplay.from_predictions(
    y_test, cascade_probabilities, name="Cascade", ax=axes[0]
)
PrecisionRecallDisplay.from_predictions(
    y_test, single_probabilities, name="D8 Balanced", ax=axes[0]
)
axes[0].set_title("Final Test Precision–Recall")
axes[0].grid(alpha=0.25)

RocCurveDisplay.from_predictions(
    y_test, cascade_probabilities, name="Cascade", ax=axes[1]
)
RocCurveDisplay.from_predictions(
    y_test, single_probabilities, name="D8 Balanced", ax=axes[1]
)
axes[1].set_title("Final Test ROC")
axes[1].grid(alpha=0.25)
fig.tight_layout()
curves_figure_path = FIGURES_DIR / "final_test_pr_roc_curves.png"
fig.savefig(curves_figure_path, dpi=170, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Görseller:")
print(" -", confusion_figure_path.relative_to(PROJECT_ROOT))
print(" -", curves_figure_path.relative_to(PROJECT_ROOT))


## 11. Final test çıktılarını kaydetme

In [ ]:
comparison_path = PATHS.metrics / "final_test_comparison.csv"
predictions_path = PREDICTIONS_DIR / "final_test_predictions.parquet"

final_test_predictions = pd.DataFrame({
    "TransactionID": id_test,
    "y_true": y_test,
    "cascade_light_probability": light_test_probabilities,
    "cascade_routed_to_heavy": routed_mask.astype(np.int8),
    "cascade_probability": cascade_probabilities,
    "cascade_prediction": cascade_predictions,
    "single_d8_probability": single_probabilities,
    "single_d8_prediction": single_predictions,
})

final_test_comparison.to_csv(comparison_path, index=False)
final_test_predictions.to_parquet(
    predictions_path, index=False, compression="zstd"
)

metadata = {
    "experiment_stage": "task_7_final_test_comparison",
    "split_version": "common_v2",
    "evaluation_dataset": "local_labeled_test_split",
    "official_competition_test_used": False,
    "test_used": True,
    "test_use_purpose": "final evaluation only",
    "models_retrained": False,
    "thresholds_retuned_on_test": False,
    "model_selection_on_test": False,
    "preselected_final_trial": PRESELECTED_FINAL_TRIAL,
    "repeat_test_evaluation_allowed": ALLOW_REPEAT_TEST_EVALUATION,
    "fixed_thresholds": fixed_thresholds.to_dict(orient="records"),
    "results": final_test_comparison.to_dict(orient="records"),
    "model_paths": {
        "cascade_light": str(CASCADE_LIGHT_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "cascade_heavy": str(CASCADE_HEAVY_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "single_heavy": str(SINGLE_MODEL_PATH.relative_to(PROJECT_ROOT)),
    },
}
with FINAL_METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
        default=lambda value: value.item()
        if isinstance(value, np.generic)
        else str(value),
    )

print("Final test çıktıları:")
for path in [
    comparison_path,
    predictions_path,
    FINAL_METADATA_PATH,
    confusion_figure_path,
    curves_figure_path,
]:
    print(" -", path.relative_to(PROJECT_ROOT))
print("\nFinal test değerlendirmesi tamamlandı; thresholdlar değiştirilmedi.")


## 12. Bellek temizliği

In [ ]:
del tree_X_test, catboost_X_test
del cascade_light_model, cascade_heavy_model, single_model
gc.collect()
print(
    "Temizlik tamamlandı. RAM:",
    f"{psutil.Process(os.getpid()).memory_info().rss / (1024**3):.2f} GB",
)


# Görev 7 tamamlanma koşulları

- [x] D8 Balanced test açılmadan önce final model adayı olarak sabitlenir.
- [x] Cascade ve tekil model aynı ortak test satırlarında değerlendirilir.
- [x] IEEE-CIS resmi etiketsiz competition test seti kullanılmaz.
- [x] Modeller yeniden eğitilmez.
- [x] Threshold değerleri test sonucuna göre değiştirilmez.
- [x] Test üzerinde model veya hiperparametre seçimi yapılmaz.
- [x] Precision, recall, F1, PR-AUC, ROC-AUC, FP ve FN raporlanır.
- [x] Gerçek tam-test ve 10K inference süreleri ölçülür.
- [x] Validation–test metrik farkları raporlanır.
- [x] Tekrar test kullanımına karşı koruma dosyası oluşturulur.

Görev 7 sonuçları incelendikten sonra Görev 8 Streamlit prototipinin kapsamı ayrıca belirlenecektir.
